In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv('IMDB Dataset.csv')
print(df.shape)

(50000, 2)


In [ ]:
import re

def clean_text(text):
    text = re.sub(r'<.*?>', '', text)
   
    text = re.sub(r'[^a-zA-Z]', ' ', text)
   
    text = text.lower()
    text = text.strip()
    return text

print("BEFORE:", df['review'][0][:200])
print("\nAFTER:", clean_text(df['review'][0])[:200])

BEFORE: One of the other reviewers has mentioned that after watching just 1 Oz episode you'll be hooked. They are right, as this is exactly what happened with me.<br /><br />The first thing that struck me abo

AFTER: one of the other reviewers has mentioned that after watching just   oz episode you ll be hooked  they are right  as this is exactly what happened with me the first thing that struck me about oz was it


In [7]:
df['label'] = df['sentiment'].map({'positive': 1, 'negative': 0})
print(df['label'].value_counts())

label
1    25000
0    25000
Name: count, dtype: int64


In [2]:
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [3]:
df['sentiment'].value_counts()

sentiment
positive    25000
negative    25000
Name: count, dtype: int64

In [4]:
df.isnull().sum()

review       0
sentiment    0
dtype: int64

In [ ]:

df['clean_review'] = df['review'].apply(clean_text)
print("Done! Total reviews cleaned:", len(df))
df[['review', 'clean_review']].head(3)

Done! Total reviews cleaned: 50000


,review,clean_review
0,One of the other reviewers has mentioned that ...,one of the other reviewers has mentioned that ...
1,A wonderful little production. <br /><br />The...,a wonderful little production the filming tec...
2,I thought this was a wonderful way to spend ti...,i thought this was a wonderful way to spend ti...


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

X_train, X_test, y_train, y_test = train_test_split(
    df['clean_review'], 
    df['label'], 
    test_size=0.2, 
    random_state=42
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

Training samples: 40000
Testing samples: 10000


In [ ]:
vectorizer = TfidfVectorizer(max_features=10000)
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

print("Shape of training data:", X_train_tfidf.shape)

Shape of training data: (40000, 10000)


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

model = LogisticRegression(max_iter=1000)
model.fit(X_train_tfidf, y_train)

y_pred = model.predict(X_test_tfidf)
accuracy = accuracy_score(y_test, y_pred)
print(f"Model Accuracy: {accuracy * 100:.2f}%")

Model Accuracy: 89.85%


In [11]:
print(classification_report(y_test, y_pred, target_names=['Negative', 'Positive']))

              precision    recall  f1-score   support

    Negative       0.91      0.89      0.90      4961
    Positive       0.89      0.91      0.90      5039

    accuracy                           0.90     10000
   macro avg       0.90      0.90      0.90     10000
weighted avg       0.90      0.90      0.90     10000



In [12]:
def predict_sentiment(review):
    clean = clean_text(review)
    vectorized = vectorizer.transform([clean])
    prediction = model.predict(vectorized)[0]
    probability = model.predict_proba(vectorized)[0]
    
    if prediction == 1:
        print(f"✅ POSITIVE ({probability[1]*100:.1f}% confident)")
    else:
        print(f"❌ NEGATIVE ({probability[0]*100:.1f}% confident)")


predict_sentiment("This movie was absolutely amazing! The acting was brilliant.")
predict_sentiment("Worst movie I have ever seen. Complete waste of time.")
predict_sentiment("It was okay, nothing special but not terrible either.")

✅ POSITIVE (95.7% confident)
❌ NEGATIVE (100.0% confident)
❌ NEGATIVE (99.8% confident)


In [13]:
import pickle

# Model aur vectorizer save karo
with open('model.pkl', 'wb') as f:
    pickle.dump(model, f)

with open('vectorizer.pkl', 'wb') as f:
    pickle.dump(vectorizer, f)

print("Model saved successfully!")

Model saved successfully!
